# 03 — GAN Augmentation

Notebook này thực hiện:
1. Tải tập đặc trưng Wavelet từ `data/processed/steel_wavelet_features.csv`
2. Chuẩn hóa dữ liệu về khoảng [-1, 1] cho GAN
3. Định nghĩa và huấn luyện mô hình GAN (Generator + Discriminator)
4. Sinh dữ liệu tổng hợp bằng Generator đã huấn luyện
5. Đánh giá chất lượng dữ liệu tổng hợp so với dữ liệu thực
6. Lưu dữ liệu tổng hợp vào `data/processed/steel_synthetic_gan.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

from src.gan_augmentation import (
    build_generator,
    build_discriminator,
    train_gan,
    generate_synthetic_samples,
)
from src.utils import load_data, save_data

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Tải dữ liệu đặc trưng Wavelet

In [ ]:
FEAT_PATH = Path('data/processed/steel_wavelet_features.csv')
df = load_data(FEAT_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Chuẩn bị dữ liệu cho GAN

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
data_array = df[numeric_cols].dropna().values
print(f'Dữ liệu thực (sau dropna): {data_array.shape}')

scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data_array)
print(f'Min: {data_scaled.min():.2f}, Max: {data_scaled.max():.2f}')

## 3. Kiến trúc GAN

In [ ]:
LATENT_DIM = 100
N_FEATURES = data_scaled.shape[1]

generator     = build_generator(latent_dim=LATENT_DIM, output_dim=N_FEATURES)
discriminator = build_discriminator(input_dim=N_FEATURES)

print('=== Generator ===')
generator.summary()
print('\n=== Discriminator ===')
discriminator.summary()

## 4. Huấn luyện GAN

In [ ]:
# Giảm epochs xuống cho demo; tăng lên 2000-5000 để huấn luyện đầy đủ
GAN_EPOCHS = 500

generator_trained, discriminator_trained = train_gan(
    data_scaled,
    latent_dim=LATENT_DIM,
    epochs=GAN_EPOCHS,
    batch_size=64,
    sample_interval=100,
)

## 5. Sinh dữ liệu tổng hợp

In [ ]:
N_SYNTHETIC = 500

synthetic_scaled = generate_synthetic_samples(
    generator_trained, n_samples=N_SYNTHETIC, latent_dim=LATENT_DIM
)
synthetic = scaler.inverse_transform(synthetic_scaled)
synthetic_df = pd.DataFrame(synthetic, columns=numeric_cols)

print(f'Synthetic data shape: {synthetic_df.shape}')
synthetic_df.describe()

## 6. So sánh phân phối: Thực vs Tổng hợp

In [ ]:
compare_cols = numeric_cols[:min(4, len(numeric_cols))]
fig, axes = plt.subplots(1, len(compare_cols), figsize=(5 * len(compare_cols), 4))
if len(compare_cols) == 1:
    axes = [axes]

real_df = pd.DataFrame(data_array, columns=numeric_cols)

for ax, col in zip(axes, compare_cols):
    ax.hist(real_df[col], bins=50, alpha=0.5, label='Real', density=True)
    ax.hist(synthetic_df[col], bins=50, alpha=0.5, label='Synthetic', density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Distribution Comparison: Real vs GAN Synthetic')
plt.tight_layout()
plt.show()

## 7. Lưu dữ liệu tổng hợp

In [ ]:
OUTPUT_PATH = Path('data/processed/steel_synthetic_gan.csv')
save_data(synthetic_df, OUTPUT_PATH)
print(f'Synthetic data saved to {OUTPUT_PATH}')